In [1]:
from langchain_community.document_loaders import PyPDFLoader,TextLoader
import os

pdf_folder = r"C:\Users\DELL\Desktop\course_prac\RAG Based Q&A bot\pdf_data"

pdf_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.pdf')]
txt_files = [f for f in os.listdir(pdf_folder) if f.lower().endswith('.txt')]

for pdf_file in pdf_files:
  pdf_path = os.path.join(pdf_folder,pdf_file)
  loader = PyPDFLoader(pdf_path)
  pages = loader.load()
  text = "\n".join([page.page_content for page in pages])
  print(f"\n--- Content of {pdf_file} (LangChain) ---\n")
  print(text[:1000])


for txt_file in txt_files:
  txt_path = os.path.join(pdf_folder,txt_file)
  loader = TextLoader(txt_path)
  pages = loader.load()
  text = "\n".join([doc.page_content for doc in docs])
  print(f"\n--- Content of {txt_file} (LangChain) ---\n")
  print(text[:1000])

C:\Users\DELL\AppData\Local\Temp\ipykernel_10376\2645141357.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,TextLoader
C:\Users\DELL\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Content of 5. Battery Management Systems.pdf (LangChain) ---

1  Battery Management Systems 
Table of Contents        Page Number 
 
1. Introduction          - 2 
2. Battery Management System       - 13 
3. Associated Components of BMS      - 17 
4. Functioning of BMS        - 25 
5. Types of BMS         - 30 
6. Wireless Distributed Battery Management System (wBMS)   - 37 
7. Adoption of AI technologies in Battery Management System  - 41 
8. Roles and Skills         - 47 
9. Global Leaders         - 55 
10. Startups           - 56 
11. References         - 57
2  Battery Management Systems 
BATTERY MANAGEMENT SYSTEMS 
1. Introduction – Battery: 
A battery is an electrochemical  device that converts chemical energy into electrical 
energy. It's made up of one or more electrochemical cells that are connected to external inputs and 
outputs. Batteries can be charged with an electric current and discharged whenever required . It is 
a device that produces electricity to provide power 

# Step-by-Step Explanation: Creating a RAG Pipeline with Gemini Embeddings and FAISS

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline for Q&A over your documents. Here’s what each major step does:

1. **Document Loading**
   - Load PDF, DOCX, and TXT files from a folder using LangChain loaders.
   - Extract the raw text from each document.

2. **Text Cleaning**
   - Clean the extracted text to remove extra spaces, newlines, and non-printable characters for better downstream processing.

3. **Chunking**
   - Split the cleaned text into smaller, overlapping chunks using `RecursiveCharacterTextSplitter`.
   - This ensures each chunk fits within the context window of the embedding model and preserves context across boundaries.

4. **Embedding Generation**
   - Use Google Gemini’s embedding model (via `GoogleGenerativeAIEmbeddings`) to convert each text chunk into a high-dimensional vector (embedding).
   - These embeddings capture the semantic meaning of each chunk for later retrieval.

5. **Vector Store Creation (FAISS)**
   - Store all embeddings in a FAISS index, which enables fast similarity search (retrieval) over the chunks.
   - Save both the FAISS index and the original text chunks for later use.

**Summary:**
- You are preparing your documents for a RAG Q&A system: load → clean → chunk → embed → store.
- Later, you can retrieve the most relevant chunks for a user query and use Gemini or another LLM to answer questions based on your documents.

## What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that combines **information retrieval** and **Large Language Models (LLMs)** to generate accurate, context-aware responses.

It first retrieves relevant information from external documents or a knowledge base based on the user's query, then provides that information as context to an LLM (such as GPT, Gemini, or Llama) to generate the final answer.

In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, Docx2txtLoader
import os

pdf_folder = r"C:\Users\DELL\Desktop\course_prac\RAG Based Q&A bot\pdf_data"

loader = DirectoryLoader(
    pdf_folder,
    glob=["*.pdf", "*.docx"],
    loader_cls=lambda path: PyPDFLoader(path) if path.lower().endswith('.pdf') else Docx2txtLoader(path)
)
documents = loader.load()

if not documents:
    print("No PDF or DOCX files found in the folder.")
else:
    for i, doc in enumerate(documents):
        print(f"\n--- Content of Document {i+1}: {doc.metadata.get('source', 'Unknown')} ---\n")
        print(doc.page_content[:1000])



--- Content of Document 1: C:\Users\DELL\Desktop\course_prac\RAG Based Q&A bot\pdf_data\5. Battery Management Systems.pdf ---

1  Battery Management Systems 
Table of Contents        Page Number 
 
1. Introduction          - 2 
2. Battery Management System       - 13 
3. Associated Components of BMS      - 17 
4. Functioning of BMS        - 25 
5. Types of BMS         - 30 
6. Wireless Distributed Battery Management System (wBMS)   - 37 
7. Adoption of AI technologies in Battery Management System  - 41 
8. Roles and Skills         - 47 
9. Global Leaders         - 55 
10. Startups           - 56 
11. References         - 57

--- Content of Document 2: C:\Users\DELL\Desktop\course_prac\RAG Based Q&A bot\pdf_data\5. Battery Management Systems.pdf ---

2  Battery Management Systems 
BATTERY MANAGEMENT SYSTEMS 
1. Introduction – Battery: 
A battery is an electrochemical  device that converts chemical energy into electrical 
energy. It's made up of one or more electrochemical cells that ar

In [3]:
import re

def clean_text(text):
  text = re.sub(r'\s+', ' ',text)
  text = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', text)
  return text.strip()

cleaned_documents = []
for i, doc in enumerate(documents):
  raw_text = doc.page_content
  cleaned = clean_text(raw_text)
  cleaned_documents.append(cleaned)
  print(f"\n--- Cleaned Content of Document {i+1}: {doc.metadata.get('source', 'Unknown')} ---\n")
  print(cleaned[:1000])




--- Cleaned Content of Document 1: C:\Users\DELL\Desktop\course_prac\RAG Based Q&A bot\pdf_data\5. Battery Management Systems.pdf ---

1 Battery Management Systems Table of Contents Page Number 1. Introduction - 2 2. Battery Management System - 13 3. Associated Components of BMS - 17 4. Functioning of BMS - 25 5. Types of BMS - 30 6. Wireless Distributed Battery Management System (wBMS) - 37 7. Adoption of AI technologies in Battery Management System - 41 8. Roles and Skills - 47 9. Global Leaders - 55 10. Startups - 56 11. References - 57

--- Cleaned Content of Document 2: C:\Users\DELL\Desktop\course_prac\RAG Based Q&A bot\pdf_data\5. Battery Management Systems.pdf ---

2 Battery Management Systems BATTERY MANAGEMENT SYSTEMS 1. Introduction – Battery: A battery is an electrochemical device that converts chemical energy into electrical energy. It's made up of one or more electrochemical cells that are connected to external inputs and outputs. Batteries can be charged with an electri

In [4]:
cleaned_documents

['1 Battery Management Systems Table of Contents Page Number 1. Introduction - 2 2. Battery Management System - 13 3. Associated Components of BMS - 17 4. Functioning of BMS - 25 5. Types of BMS - 30 6. Wireless Distributed Battery Management System (wBMS) - 37 7. Adoption of AI technologies in Battery Management System - 41 8. Roles and Skills - 47 9. Global Leaders - 55 10. Startups - 56 11. References - 57',
 "2 Battery Management Systems BATTERY MANAGEMENT SYSTEMS 1. Introduction – Battery: A battery is an electrochemical device that converts chemical energy into electrical energy. It's made up of one or more electrochemical cells that are connected to external inputs and outputs. Batteries can be charged with an electric current and discharged whenever required . It is a device that produces electricity to provide power for electronic devices, cars, etc. Batteries are divided into primary batteries, which can only be used once, such as dry cell batteries, and secondary batteries, 

In [5]:
len(cleaned_documents)

108

In [6]:
cleaned_documents[0]

'1 Battery Management Systems Table of Contents Page Number 1. Introduction - 2 2. Battery Management System - 13 3. Associated Components of BMS - 17 4. Functioning of BMS - 25 5. Types of BMS - 30 6. Wireless Distributed Battery Management System (wBMS) - 37 7. Adoption of AI technologies in Battery Management System - 41 8. Roles and Skills - 47 9. Global Leaders - 55 10. Startups - 56 11. References - 57'

In [7]:
len(cleaned_documents[0])

410

In [85]:

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

char_splitter = CharacterTextSplitter(separator = "\n", chunk_size = 1000, chunk_overlap = 200)
char_chunks = []
for doc in cleaned_documents:
  char_chunks.extend(char_splitter.split_text(doc))
print(f"characterTextSplitter produced {len(char_chunks)} chunks.")

rec_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150)
rec_chunks = []
for doc in cleaned_documents:
  rec_chunks.extend(rec_splitter.split_text(doc))
print(f"RecursiveCharacterTextSplitter produced {len(rec_chunks)} chunks.")

for i, chunk in enumerate(rec_chunks[:3]):
  print(f"\n---Recursive chunk {i+1}---\n{chunk[:500]}")



characterTextSplitter produced 108 chunks.
RecursiveCharacterTextSplitter produced 157 chunks.

---Recursive chunk 1---
1 Battery Management Systems Table of Contents Page Number 1. Introduction - 2 2. Battery Management System - 13 3. Associated Components of BMS - 17 4. Functioning of BMS - 25 5. Types of BMS - 30 6. Wireless Distributed Battery Management System (wBMS) - 37 7. Adoption of AI technologies in Battery Management System - 41 8. Roles and Skills - 47 9. Global Leaders - 55 10. Startups - 56 11. References - 57

---Recursive chunk 2---
2 Battery Management Systems BATTERY MANAGEMENT SYSTEMS 1. Introduction – Battery: A battery is an electrochemical device that converts chemical energy into electrical energy. It's made up of one or more electrochemical cells that are connected to external inputs and outputs. Batteries can be charged with an electric current and discharged whenever required . It is a device that produces electricity to provide power for electronic devices, 

# Text Splitters in LangChain

## 1. CharacterTextSplitter

### What is it?

`CharacterTextSplitter` is the simplest text splitting method. It divides text into chunks based only on the **number of characters**, without considering sentences or paragraphs.

### How it works

- Splits text using a specified separator (optional).
- Creates chunks of a fixed size.
- Adds overlap between consecutive chunks (if specified).

### Pros

- ✅ Simple and fast
- ✅ Easy to configure
- ✅ Suitable for code, logs, and raw text

### Cons

- ❌ May split words or sentences in the middle
- ❌ Doesn't preserve context well
- ❌ Lower semantic quality for RAG

---

# 2. RecursiveCharacterTextSplitter

### What is it?

`RecursiveCharacterTextSplitter` is a smarter text splitter that tries to preserve the natural structure of the text while creating chunks.

### How it works

It recursively splits text using the following separators:

```python
["\n\n", "\n", ".", "!", "?", ",", " ", ""]
```

It first tries to split by:

1. Paragraphs
2. Lines
3. Sentences
4. Words
5. Characters (last option)

This keeps related information together as much as possible.

### Pros

- ✅ Preserves paragraphs and sentences
- ✅ Produces meaningful chunks
- ✅ Better context for embeddings
- ✅ Best choice for RAG applications

### Cons

- ❌ Slightly slower than CharacterTextSplitter
- ❌ More complex internally

---

# CharacterTextSplitter vs RecursiveCharacterTextSplitter

| Feature | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|---------|-----------------------|--------------------------------|
| Split Method | Fixed character count | Recursive, hierarchical splitting |
| Preserves Sentences | ❌ No | ✅ Yes |
| Semantic Context | ❌ Low | ✅ High |
| Speed | Faster | Slightly slower |
| Best For | Code, Logs, Raw Text | PDFs, Articles, Documents, RAG |

---

# Why does RecursiveCharacterTextSplitter create more chunks?

- **CharacterTextSplitter** splits text strictly based on the specified separator and character limit, even if it cuts sentences.
- **RecursiveCharacterTextSplitter** tries to preserve paragraphs, sentences, and words. If a chunk is still too large, it recursively splits further.

As a result, **RecursiveCharacterTextSplitter often produces more chunks, but they are more meaningful and provide better context for retrieval.**

---

# Which one should I use?

- **CharacterTextSplitter** → Best for code files, logs, and simple text.
- **RecursiveCharacterTextSplitter** → Best for RAG pipelines, PDFs, document Q&A, and summarization.

> **Recommendation:** For most RAG projects, use **RecursiveCharacterTextSplitter** because it creates contextually meaningful chunks, leading to better retrieval and answer generation.

# JSON Character Splitter

## What is it?

A **JSON Character Splitter** is used to divide large JSON files into smaller chunks while trying to preserve the JSON structure. This helps LLMs process JSON data within token limits.

---

# Why not use a normal CharacterTextSplitter?

A normal `CharacterTextSplitter` treats JSON as plain text and may split in the middle of:

- Keys
- Values
- Objects
- Arrays

This can create **invalid JSON** and reduce embedding quality.

---

# Types of JSON Chunking

## 1. Character-Based JSON Splitter

- Treats JSON as plain text.
- Splits based on character count.
- Fast but may break JSON syntax.

**Best for:** Small JSON files or simple logs.

---

## 2. Structural JSON Splitter (Recommended)

- Parses JSON into Python objects (`dict`/`list`).
- Splits JSON by complete objects.
- Keeps JSON valid and meaningful.

**Best for:** JSON datasets and RAG pipelines.

---

## 3. RecursiveCharacterTextSplitter for JSON

- Uses JSON-aware separators such as:

```python
["},", "],", "}", "]", ","]
```

- Attempts to split at object or array boundaries before splitting by characters.
- Produces more meaningful chunks than simple character splitting.

**Best for:** Large or nested JSON files.

---

# Comparison

| Splitter | Best For | Preserves JSON |
|----------|----------|----------------|
| Character-Based | Small JSON, Logs | ❌ No |
| Structural JSON Splitter | JSON Objects, Datasets | ✅ Yes |
| RecursiveCharacterTextSplitter | Large/Nested JSON | ✅ Mostly |

---

# Which one should I use?

- **Character-Based Splitter** → Small JSON files and logs.
- **Structural JSON Splitter** → Best choice for JSON datasets and RAG.
- **RecursiveCharacterTextSplitter** → Best for large or nested JSON documents.

In [86]:
char_chunks

['1 Battery Management Systems Table of Contents Page Number 1. Introduction - 2 2. Battery Management System - 13 3. Associated Components of BMS - 17 4. Functioning of BMS - 25 5. Types of BMS - 30 6. Wireless Distributed Battery Management System (wBMS) - 37 7. Adoption of AI technologies in Battery Management System - 41 8. Roles and Skills - 47 9. Global Leaders - 55 10. Startups - 56 11. References - 57',
 "2 Battery Management Systems BATTERY MANAGEMENT SYSTEMS 1. Introduction – Battery: A battery is an electrochemical device that converts chemical energy into electrical energy. It's made up of one or more electrochemical cells that are connected to external inputs and outputs. Batteries can be charged with an electric current and discharged whenever required . It is a device that produces electricity to provide power for electronic devices, cars, etc. Batteries are divided into primary batteries, which can only be used once, such as dry cell batteries, and secondary batteries, 

In [87]:
rec_chunks

['1 Battery Management Systems Table of Contents Page Number 1. Introduction - 2 2. Battery Management System - 13 3. Associated Components of BMS - 17 4. Functioning of BMS - 25 5. Types of BMS - 30 6. Wireless Distributed Battery Management System (wBMS) - 37 7. Adoption of AI technologies in Battery Management System - 41 8. Roles and Skills - 47 9. Global Leaders - 55 10. Startups - 56 11. References - 57',
 "2 Battery Management Systems BATTERY MANAGEMENT SYSTEMS 1. Introduction – Battery: A battery is an electrochemical device that converts chemical energy into electrical energy. It's made up of one or more electrochemical cells that are connected to external inputs and outputs. Batteries can be charged with an electric current and discharged whenever required . It is a device that produces electricity to provide power for electronic devices, cars, etc. Batteries are divided into primary batteries, which can only be used once, such as dry cell batteries, and secondary batteries, 

In [88]:
len(char_chunks)

108

In [89]:
len(rec_chunks)

157

In [90]:
rec_chunks

['1 Battery Management Systems Table of Contents Page Number 1. Introduction - 2 2. Battery Management System - 13 3. Associated Components of BMS - 17 4. Functioning of BMS - 25 5. Types of BMS - 30 6. Wireless Distributed Battery Management System (wBMS) - 37 7. Adoption of AI technologies in Battery Management System - 41 8. Roles and Skills - 47 9. Global Leaders - 55 10. Startups - 56 11. References - 57',
 "2 Battery Management Systems BATTERY MANAGEMENT SYSTEMS 1. Introduction – Battery: A battery is an electrochemical device that converts chemical energy into electrical energy. It's made up of one or more electrochemical cells that are connected to external inputs and outputs. Batteries can be charged with an electric current and discharged whenever required . It is a device that produces electricity to provide power for electronic devices, cars, etc. Batteries are divided into primary batteries, which can only be used once, such as dry cell batteries, and secondary batteries, 

# Embeddings

## What are Embeddings?

Embeddings are **numerical vector representations** of text that capture its semantic meaning. Words or sentences with similar meanings have similar embedding vectors.

**Example:**

```
Science → [0.01, 0.25, 0.78, ..., 0.43]
```

These vectors are used for:

- Semantic Search
- RAG (Retrieval-Augmented Generation)
- Similarity Search
- Clustering
- Recommendation Systems

---

# Popular Embedding Models

- Google Generative AI Embeddings
- OpenAI Embeddings
- Hugging Face Embeddings
- Cohere Embeddings
- Amazon Titan Embeddings
- Ollama Embeddings
- LlamaCpp Embeddings

Each model converts text into vectors of different dimensions.

---

# Google Generative AI Embeddings

The **Gemini Embedding Model (`gemini-embedding-001`)** generates vectors of:

- **3072 dimensions (default)**
- **1536 dimensions (optional)**
- **768 dimensions (optional)**

Smaller dimensions reduce storage and computation but may slightly reduce embedding quality.

---

# OpenAI Embedding Models

| Model | Dimensions | Description |
|--------|-----------:|------------|
| **text-embedding-ada-002** | 1536 | Older embedding model |
| **text-embedding-3-small** | 1536 | Faster and cheaper |
| **text-embedding-3-large** | 3072 | More accurate and expressive |

---

# text-embedding-3-small vs text-embedding-3-large

| Feature | 3-small | 3-large |
|---------|---------|---------|
| Dimensions | 1536 | 3072 |
| Accuracy | Good | High |
| Speed | Faster | Slower |
| Cost | Lower | Higher |
| Storage | Less | More |

---

# When to Use Which?

### Use **text-embedding-3-small**

- Semantic Search
- FAQ Search
- Topic Clustering
- Duplicate Detection
- Large-scale retrieval with lower cost

### Use **text-embedding-3-large**

- RAG Applications
- Document Q&A
- Knowledge Retrieval
- Long-context reasoning
- High-accuracy semantic search

---

# Key Takeaways

- Embeddings convert text into vectors.
- Similar text has similar vectors.
- Higher dimensions generally provide richer semantic representations.
- **3-small** → Faster and cheaper.
- **3-large** → More accurate and recommended for RAG.
- **Gemini Embeddings** support **3072, 1536, and 768** dimensions.

In [91]:
import numpy as np
from langchain_pinecone import PineconeVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
import os
import getpass

In [92]:
from dotenv import load_dotenv
import os

# Load variables from .env file
load_dotenv()

# Read the API key
api_key = os.getenv("api_key")



In [93]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3965.10it/s]


In [17]:
import sys
print(sys.version)

3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]


In [18]:
import sentence_transformers
print(sentence_transformers.__version__)

5.6.0


In [94]:
from langchain_community.docstore.document import Document
rec_documents = [Document(page_content=chunk) for chunk in rec_chunks]

In [95]:
rec_embeddings = embedding_model.embed_documents([doc.page_content for doc in rec_documents])

In [96]:
rec_embeddings_np = np.array(rec_embeddings).astype('float32')

In [97]:
dimension = rec_embeddings_np.shape[1]


Most Common Vector Databases for RAG Projects

| Database                                       | Description                                                                 | Deployment           | Notes                                                                     |
| ---------------------------------------------- | --------------------------------------------------------------------------- | -------------------- | ------------------------------------------------------------------------- |
| **Pinecone**                                   | Fully managed cloud-native vector DB built for RAG and real-time retrieval. | Cloud (SaaS)         | ⚡Fast, serverless, supports hybrid filtering, easy LangChain integration. |
| **Weaviate**                                   | Open-source and cloud vector DB with hybrid (BM25 + vector) search.         | Cloud or Self-hosted | 💡Supports GraphQL, modular schema, and metadata-rich queries.            |
| **FAISS**                                      | Facebook AI Similarity Search — library, not a server.                      | Local / Self-hosted  | 🧮 Extremely fast for small-to-mid datasets; ideal for prototypes.        |
| **Chroma**                                     | Open-source, simple-to-use local vector DB (default for LangChain).         | Local / Self-hosted  | 🧩 Great for POCs or small-scale use; not ideal for high concurrency.     |
| **Milvus**                                     | Scalable distributed vector DB for big data workloads.                      | Cloud / Self-hosted  | ⚙️ Handles billions of embeddings; supports hybrid search.                |
| **Qdrant**                                     | Rust-based vector DB optimized for performance and memory use.              | Cloud / Self-hosted  | 🚀 Fast, lightweight, good for real-time APIs.                            |
| **Redis Vector Similarity Search (Redis VSS)** | Add-on to Redis for vector search.                                          | Cloud / Self-hosted  | 🔴 Great if you already use Redis for caching / session storage.          |
| **Azure Cognitive Search**                     | Microsoft’s managed semantic search service (supports vector search).       | Cloud                | ☁️ Works natively with Azure OpenAI and enterprise data.                  |
| **Elasticsearch + Vector Plugin**              | Traditional search engine extended for vector similarity.                   | Cloud / Self-hosted  | 🔍 Useful for hybrid keyword + semantic search.          


 Recommendation by Use Case

| Use Case                                | Recommended Vector DB                                          |
| --------------------------------------- | -------------------------------------------------------------- |
| **Enterprise RAG / Real-time chatbots** | 🥇 **Pinecone** or **Azure Cognitive Search**                  |
| **Open-source, self-hosted real-time**  | 🥇 **Qdrant** or **Weaviate**                                  |
| **Big-data / billion-scale embeddings** | **Milvus**                                                     |
| **Small apps or local dev (POC)**       | **Chroma** or **FAISS**                                        |
| **Hybrid keyword + semantic search**    | **Weaviate**, **Elasticsearch**, or **Azure Cognitive Search** |
| **Existing Redis infrastructure**       | **Redis Vector Search**                                        |


In [23]:
from pinecone import Pinecone

pc = Pinecone(api_key=api_key)

print(pc)

In [98]:
from pinecone import Pinecone
import os
pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))

In [99]:
dimension = rec_embeddings_np.shape[1]
print(dimension)

384


In [100]:
from pinecone import ServerlessSpec

index_name = "rag-index"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index Ready!")

Index Ready!


In [101]:
index = pc.Index(index_name)

print(index)

In [102]:
vectors = []

for i, (embedding, chunk) in enumerate(zip(rec_embeddings_np, rec_chunks)):
    vectors.append({
        "id": str(i),
        "values": embedding.tolist(),
        "metadata": {
            "text": chunk
        }
    })

In [103]:
index.upsert(vectors=vectors)

{'upserted_count': 157}

In [104]:
stats = index.describe_index_stats()
print(stats)

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 178}},
 'total_vector_count': 178,
 'vector_type': 'dense'}


In [105]:
query = "What is Battery Management System?"
query_embedding = embedding_model.embed_query(query)

In [106]:
results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

In [107]:
for match in results["matches"]:
    print(match["metadata"]["text"])

13 Battery Management Systems 2. BATTERY MANAGEMENT SYSTEM (BMS) Battery management device in electric vehicles is a mainly designed electronic circuit that ensures the safety and stability of these battery packs.
31 Battery Management Systems a) Centralized Battery Management Systems b) Distributed Battery Management Systems ii. Based on Voltage(16) b. High Voltage Battery Management Systems c. Low Voltage Battery Management Systems Centralized Battery Management Systems(12)(17)(18):  It has a central BMS in the battery pack assembly.  All the battery packages are connected to the central BMS directly.  Advantages: o It is more compact, and it tends to be the most economical since there is only one BMS.
14 Battery Management Systems Electric vehicles run on rechargeable battery packs that are made of multiple cell modules arranged in a series and parallel. These battery packs produce several hu ndred volts of electricity. Various functions inside the car are dependent on them. That

In [108]:
def build_context_passage(results):
    passages = []

    for match in results["matches"]:
        passages.append(match["metadata"]["text"])

    return "\n\n".join(passages)

In [109]:
context = build_context_passage(results)

print("Retrieved Context:\n")
print(context)

Retrieved Context:

13 Battery Management Systems 2. BATTERY MANAGEMENT SYSTEM (BMS) Battery management device in electric vehicles is a mainly designed electronic circuit that ensures the safety and stability of these battery packs.

31 Battery Management Systems a) Centralized Battery Management Systems b) Distributed Battery Management Systems ii. Based on Voltage(16) b. High Voltage Battery Management Systems c. Low Voltage Battery Management Systems Centralized Battery Management Systems(12)(17)(18):  It has a central BMS in the battery pack assembly.  All the battery packages are connected to the central BMS directly.  Advantages: o It is more compact, and it tends to be the most economical since there is only one BMS.

14 Battery Management Systems Electric vehicles run on rechargeable battery packs that are made of multiple cell modules arranged in a series and parallel. These battery packs produce several hu ndred volts of electricity. Various functions inside the car are d

In [206]:

def build_prompt(context,question):
  prompt = (f'''
    "You are an AI assistant that answers questions ONLY from the provided context."
            
    "Instructions:" 
    "1.Use only the given context."
    "2.Do not use outside knowledge."
    "3.If the answer is not found in the context, say:" 
      "I couldn't find the answer in the provided documents."
    "4.Give detailed explanation."
    "5.Include every relevant point from the context."
    "6.If multiple documents discuss the topic, combine the information."
    "7.Use headings and Bullet points whether appropriate."
    "8.Explain each point instead of listing only keywords."
    "9.Do not infer, assume, or generate information that is not explicitly present in the provided context."
    "10.If the context partially answers the question, answer only the available part and clearly mention what information is missing."
    "11.Do not use your general knowledge, even if you know the answer."
    "12.Every statement in the answer must be directly supported by the provided context."
    "13.Do not add examples, comparisons, explanations, or conclusions unless they appear in the context."
    "14.If multiple retrieved chunks contain relevant information, combine them into a single coherent answer without adding new information."
    "15.If the answer cannot be completely answered from the context, state:"
     "The provided documents do not contain enough information to fully answer this question."
    "16. Do not answer using information that is not present in the retrieved context, even if it exists elsewhere in the original documents or in your own knowledge."

    f"Context:\n{context}\n\n"
    f"Question: {question}\n"
    "Answer:"
    '''
  )
  return prompt
  

In [207]:
from dotenv import load_dotenv
import os

load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [208]:
def connect_to_pinecone():
    pc = Pinecone(api_key= pinecone_api_key)
    return pc.Index(index_name)

In [209]:
index = connect_to_pinecone()

In [210]:
user_question = "Explain the Kalman Filter and its role in Sensor Fusion." 

In [211]:
query_embedding = embedding_model.embed_query(user_question)

In [212]:
def retrieve_top_k_chunks(query_embedding, index, k=5):

    results = index.query(
        vector=query_embedding,
        top_k=k,
        include_metadata=True
    )

    return results

In [213]:
top_chunks = retrieve_top_k_chunks(
    query_embedding,
    index,
    k=5
)

In [214]:
context = build_context_passage(top_chunks)

In [215]:
from google import genai
from dotenv import load_dotenv
import os

In [216]:
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

In [217]:
client = genai.Client(api_key=gemini_api_key)


In [227]:
def generate_answer_with_prompt_engineering(context, question):

    prompt = build_prompt(context, question)

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.2,
            "max_output_tokens": 2048
        }
    )

    return response.text

In [228]:
answer = generate_answer_with_prompt_engineering(
    context,
    user_question
)

print(answer)

The Kalman Filter is an algorithm used for estimating the state of a system from measured data. It was developed by the Hungarian engineer Rudolf Kalman.

### Kalman Filter Algorithm

The algorithm operates in a two-step process:
*   **Prediction:** The first step involves predicting the state of the system.
*   **Refinement:** The second step utilizes noisy measurements to refine the estimate of the system's state.

There are several variants of the original Kalman filter.

### Applications of Kalman Filters

Kalman filters are widely employed in applications that depend on estimation, including:
*   Computer vision
*   Guidance and navigation systems
*   Econometrics
*   Signal processing

### Variants of Kalman Filters

*   **Extended Kalman Filter (EKF):** The EKF is mentioned in contrast to the Unscented Kalman Filter. Attempts to reduce the computational cost of the EKF through linearization can make its performance unstable.
*   **Unscented Kalman Filter (UKF):** The UKF has gai

In [220]:
print(context)

Creamcollar ADAS / AdapƟve Cruise Control / Sensor Fusion 9 What Is a Kalman Filter? The Kalman filter is an algorithm that estimates the state of a system from measured data. It was primarily developed by the Hungarian engineer Rudolf Kalman, for whom the filter is named. The filter’s algorithm is a two-step process: the first step predicts the state of the system, and the second step uses noisy measurements to refine the estimate of system state. There are now several variants of the original Kalman filter. These filters are widely used for applications that rely on estimation, including computer vision, guidance and navigation systems, econometrics, and signal processing(4). Use cases and Sensor requirements(15):

Creamcollar ADAS / AdapƟve Cruise Control / Sensor Fusion 49 References: 1. https://www.kpit.com/insights/achieve-sensor-fusion-for-autonomous-driving-adas- vehicles/ 2. file:///C:\Creamcollar\Creamcollar\ADAS\ACC\Sensor%20fusion\sensor%20fusion.p df 3. Centralised and Dec